In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

### **Importing Prerequisites**

In [0]:
%run "/Workspace/Users/0bcr9999@gmail.com/FMGC-DATAENGINEERING-PROJECT/notebooks/setup/3. utilities"

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data_source", "", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog, data_source)

### **Read Data from Bronze Layer**

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")

In [0]:
display(df_bronze)

### **Transformations**

**1. Normalize `month` column**

In [0]:
df_bronze.select("month").distinct().show()

In [0]:
date_format = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"]

df_silver = (
    df_bronze.withColumn(
        "month",
        F.coalesce(
            F.try_to_date(F.col("month"), "yyyy/MM/dd"),
            F.try_to_date(F.col("month"), "dd/MM/yyyy"),
            F.try_to_date(F.col("month"), "yyyy-MM-dd"),
            F.try_to_date(F.col("month"), "dd-MM-yyyy")
        )
    )
)

In [0]:
df_silver.select("month").distinct().show()

**2. Handling `gross_price` issues**

In [0]:
df_silver.select("gross_price").distinct().show()

In [0]:
df_silver = (
    df_silver.withColumn(
        "gross_price",
        F.when(F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$'), F.abs(F.col("gross_price").cast("double")))
        .otherwise(0)
    )
)

In [0]:
df_silver.select("gross_price").distinct().show()

**3. Enrich product code using products table for the associated products**

In [0]:
df_products = spark.read.table("fmcg.silver.products")
df_joined = df_silver.join(df_products.select("product_code", "product_id"), on = "product_id", how = "inner")

In [0]:
display(df_joined)

**4. Re-ordering columns**

In [0]:
df_joined = df_joined.select("product_id", "product_code", "month", "gross_price", "dataload_ts","file_name", "file_size")

In [0]:
display(df_joined.limit(5))

### **writing dataframe to silver table**

In [0]:
df_joined.write \
  .format("delta") \
    .option("delta.enableChangeDataFeed", True) \
      .option("mergeSchema", True) \
        .mode("overwrite") \
          .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")